
# 37 WT-plus 因子工具链：从 RD-Agent 到护栏

> **本章要解决的问题**：WT-plus 不是一个“因子软件”，而是一套执行平台；因子研究由 RD-Agent、KunQuant、TA-Lib、Alphalens、fincore、Optuna 和几道护栏拼起来。它们各自做什么、按什么顺序接、在哪里容易出错？

## 本章导读

这一章把散落在 11、14、15、32 里的工具名收拢成一张图：

1. WT-plus 是什么：WonderTrader-plus + wtpy-plus；
2. 研究循环：RD-Agent + Optuna；
3. 计算：KunQuant + TA-Lib；
4. 初筛与组合：Alphalens-reloaded + fincore；
5. 护栏：Great Expectations、MLflow、DSR/PBO；
6. 回放：WT-plus 原生回放。

### 学习目标

- 能说清 WT-plus 与因子工具链的边界：谁是执行平台，谁是研究工具。
- 能画出 RD-Agent 的“假设→代码→回测→反馈”循环，并指出 Optuna 在循环里的位置。
- 能解释 KunQuant 的批编译、8 的倍数约束，以及 TA-Lib 的基线作用。
- 能用 Alphalens 的 IC/ICIR/分组/换手做第一道初筛。
- 能解释 fincore 在组合绩效/归因中的位置。
- 能说出 Great Expectations、MLflow、DSR/PBO 各自防什么。
- 能写出 WT-plus 原生回放与普通回测的区别。

### 先修

第12章的因子流水线；第14章的回放；第20章的多重检验；会 pandas/numpy。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="37 WT-plus 因子工具链：从 RD-Agent 到护栏" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>37 WT-plus 因子工具链：从 RD-Agent 到护栏｜知识图</title>
<desc>本章主线：研究循环、计算、初筛、组合、护栏、回放各管一段；顺序错了结论不可信。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a37" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao37" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag37" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">37</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">WT-plus 因子工具链：从 RD-Agent 到护栏</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：研究循环、计算、初筛、组合、护栏、回放各管一段；顺序错了结论不可信。</text>
<rect x="41" y="128" width="165" height="148" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="123.5" y="188.4" font-size="13" fill="#1B3A5C" text-anchor="middle" font-weight="bold">① 生成</text>
<text x="123.5" y="206.9" font-size="8.8" fill="#6E7B8B" text-anchor="middle">RD-Agent + Optuna</text>
<text x="123.5" y="220.2" font-size="8.8" fill="#6E7B8B" text-anchor="middle">假设→代码→回测→反馈</text>
<rect x="224" y="128" width="165" height="148" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="306.5" y="188.4" font-size="13" fill="#1B3A5C" text-anchor="middle" font-weight="bold">② 计算 / 初筛</text>
<text x="306.5" y="206.9" font-size="8.8" fill="#6E7B8B" text-anchor="middle">KunQuant + TA-Lib</text>
<text x="306.5" y="220.2" font-size="8.8" fill="#6E7B8B" text-anchor="middle">Alphalens：IC / 分组 / 换手</text>
<rect x="407" y="128" width="165" height="148" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="489.5" y="188.4" font-size="13" fill="#1B3A5C" text-anchor="middle" font-weight="bold">③ 护栏</text>
<text x="489.5" y="206.9" font-size="8.8" fill="#6E7B8B" text-anchor="middle">Great Expectations</text>
<text x="489.5" y="220.2" font-size="8.8" fill="#6E7B8B" text-anchor="middle">MLflow · DSR / PBO</text>
<rect x="590" y="128" width="165" height="148" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="672.5" y="188.4" font-size="13" fill="#1B3A5C" text-anchor="middle" font-weight="bold">④ 组合 / 回放</text>
<text x="672.5" y="206.9" font-size="8.8" fill="#6E7B8B" text-anchor="middle">fincore 组合归因</text>
<text x="672.5" y="220.2" font-size="8.8" fill="#6E7B8B" text-anchor="middle">WT-plus 原生回放</text>
<path d="M 206 202 L 224 202" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a37)"/>
<path d="M 389 202 L 407 202" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a37)"/>
<path d="M 572 202 L 590 202" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a37)"/>
<path d="M 489.5 128 Q 300 92 123.5 128" fill="none" stroke="#E07B39" stroke-width="1.4" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 4" marker-end="url(#ao37)"/>
<text x="305" y="88" font-size="9" fill="#E07B39" text-anchor="middle">研究循环：护栏结果反馈给生成</text>
<rect x="41" y="298" width="718" height="82" fill="#E6EFF7" rx="14"/>
<rect x="57" y="312" width="4" height="8" fill="#E07B39" rx="1"/>
<text x="67" y="320" font-size="9.4" fill="#1B3A5C">WT-plus = 执行平台：撮合、回放、账户与持仓语义</text>
<rect x="57" y="334" width="4" height="8" fill="#E07B39" rx="1"/>
<text x="67" y="342" font-size="9.4" fill="#1B3A5C">因子研究 = RD-Agent + KunQuant / TA-Lib + Alphalens + fincore + 护栏</text>
<rect x="57" y="356" width="4" height="8" fill="#E07B39" rx="1"/>
<text x="67" y="364" font-size="9.4" fill="#6E7B8B">顺序：数据校验 → 多重检验（DSR/PBO）→ 追踪（MLflow）→ 原生回放</text>
</svg>

> **图**：本章主线：研究循环、计算、初筛、组合、护栏、回放各管一段；顺序错了结论不可信。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：因子研究员、WT-plus 平台维护、策略开发、风控。
- **输入**：研报/数据、因子表达式、交易日历、成本/账户配置。
- **输出**：因子报告、DSR/PBO 结论、组合归因、原生回放记录。
- **在流程的哪一步**：从“读论文”到“上账户”的整条流水线。
- **失败模式**：把 RD-Agent 的候选当结论、跳过 DSR/PBO、用普通回测代替 WT-plus 回放。
- **最小工作流**：数据校验 → RD-Agent 循环 → KunQuant/TA-Lib → Alphalens → DSR/PBO → fincore → WT-plus 回放。

---

## 1. WT-plus：执行平台，不是因子研究器

### 1.1 两个仓库

- **WonderTrader-plus**：C++ 核心，负责撮合、回放、账户/持仓、执行语义。  
  <https://github.com/newbigdeng/WonderTrader-plus>
- **wtpy-plus**：Python 封装，负责策略/数据/回放接口。  
  <https://github.com/newbigdeng/wtpy-plus>

WT = 你改造后的 WonderTrader；**plus** 是改造版。它回答“如果当时真的这么跑，账会怎样”，不回答“这个因子有没有 alpha”。

### 1.2 工具链全景

| 环节 | 工具 | 作用 | 产物 |
|---|---|---|---|
| 数据入口 | Great Expectations | 字段/时序/异常值断言 | 校验报告 |
| 研究循环 | RD-Agent | 假设→代码→回测→反馈 | 候选表达式 |
| 超参 | Optuna | 循环内 TPE 搜索 | 参数/试验记录 |
| 计算 | KunQuant | 表达式批量编译 C++ | 因子面板 |
| 基线 | TA-Lib | 150+ 标准指标 | 基线因子 |
| 初筛 | Alphalens-reloaded | IC/ICIR/分组/换手 | 单因子报告 |
| 护栏 | DSR/PBO | 多重检验/过拟合 | 校正后显著性 |
| 组合 | fincore | 绩效/风险/归因 | 组合报告 |
| 追踪 | MLflow | 版本/参数/指标 | 可复现记录 |
| 回放 | WT-plus | 撮合/费用/持仓 | 账户级回放 |

**关键边界**：WT-plus 只管执行与账户；因子研究是前面那条 Python 链。

---

## 2. RD-Agent + Optuna：研究循环

### 2.1 循环

RD-Agent 做的是：

1. 读研报/数据，提出假设；
2. 生成因子表达式/代码；
3. 回测/验证；
4. 把反馈喂回下一轮。

它不是“自动找到 alpha”，而是“自动扩大搜索”。搜索越大，多重检验问题越重，所以后面必须有 DSR/PBO。

### 2.2 Optuna 的位置

- LLM 改**结构**（表达式/逻辑）；
- Optuna 调**参数**（窗口、阈值、超参）；
- 两者分工，不要把参数搜索也交给 LLM。

### 2.3 代码：循环的边际收益



In [2]:

import numpy as np

rng = np.random.default_rng(3701)
trials = np.arange(1, 101)
best_ic = 0.02 + 0.03 * (1 - np.exp(-trials / 25)) + rng.normal(0, 0.002, size=len(trials))
print("第 1/10/50/100 轮最佳 IC", [round(float(best_ic[i]), 3) for i in [0, 9, 49, 99]])
print("边际提升（最后 10 轮）", round(float(best_ic[-1] - best_ic[-11]), 3))


第 1/10/50/100 轮最佳 IC [0.023, 0.031, 0.049, 0.048]
边际提升（最后 10 轮） -0.003



**读数**：最佳 IC 先快速上升，后面趋平；最后 10 轮的提升已经很小，甚至为负。**循环不是越久越好，边际收益递减时要停。**

**常见错误**：

1. 把“最佳 IC”当“可交易”。
2. 不记录试验次数。
3. 让 LLM 同时改结构和参数，无法归因。

---

## 3. KunQuant + TA-Lib：计算层

### 3.1 KunQuant

- 把表达式批量编译成优化 C++；比 pandas 快百倍级；
- **输入数量必须是 8 的倍数**（8、16、24……）；不足要补齐；
- 它加速“算”，不保证“算对”；仍要用手算样本核对。

### 3.2 TA-Lib

- 150+ 经典指标/K 线形态；
- 作为标准算子库/基线因子；
- 基线不是 alpha，但能帮你判断“新因子是否只是 MACD 的变体”。

### 3.3 代码：8 的倍数



In [3]:

for n in [100, 104, 112, 120]:
    padded = ((n + 7) // 8) * 8
    print(f"输入 {n}，是否 8 的倍数 {n % 8 == 0}，补齐到 {padded}")


输入 100，是否 8 的倍数 False，补齐到 104
输入 104，是否 8 的倍数 True，补齐到 104
输入 112，是否 8 的倍数 True，补齐到 112
输入 120，是否 8 的倍数 True，补齐到 120



**读数**：100/104/112/120 里只有 104/112/120 是 8 的倍数；100 要补到 104。**批量计算前先对齐，否则 KunQuant 会报错或截断。**

**常见错误**：

1. 直接喂任意长度。
2. 补齐用 0 填充，污染因子。
3. 不抽一行手算核对。

---

## 4. Alphalens + fincore + DSR/PBO：从初筛到组合

### 4.1 Alphalens-reloaded

对单因子做：

- IC/ICIR；
- 分组收益单调性；
- 换手率；
- 行业/市值暴露（可选）。

它是“第一道过滤器”，不是最终结论。

### 4.2 DSR/PBO

RD-Agent 试了很多轮，必须校正：

- **DSR**：Deflated Sharpe Ratio，按试验次数/样本长度调整；
- **PBO**：Probability of Backtest Overfitting，用组合式交叉验证估计“冠军是过拟合”的概率。

位置：**Alphalens 之后、fincore 之前**。

### 4.3 fincore

- 多因子组合的绩效、风险、归因；
- Brinson/因子归因、150+ 指标；
- Python 3.11+；用 `fincore.metrics`、`fincore.attribution` 等领域模块，不要找 `fincore.alphalens` 兼容别名。

### 4.4 代码：IC/ICIR 与 DSR 门槛



In [4]:

import numpy as np

rng = np.random.default_rng(3702)
ic = rng.normal(0.03, 0.08, size=120)
icir = ic.mean() / ic.std(ddof=1)
t_stat = ic.mean() / (ic.std(ddof=1) / np.sqrt(len(ic)))
print("IC 均值", round(ic.mean(), 4), "ICIR", round(icir, 3), "t 值", round(t_stat, 2))

m, T = 100, 1.0
threshold = np.sqrt(2 * np.log(m) / T)
print("100 轮试验的近似 DSR 门槛（年化 Sharpe）", round(threshold, 3))


IC 均值 0.044 ICIR 0.567 t 值 6.21
100 轮试验的近似 DSR 门槛（年化 Sharpe） 3.035



**读数**：IC 4.4%、ICIR 0.567、t 值约 6.21；但 100 轮试验的“最大 Sharpe”门槛约 3.03，不能只看单次 t 值。

**常见错误**：

1. 只看 IC 均值，不看 ICIR/换手。
2. 不做 DSR/PBO。
3. 把 fincore 的组合归因当“因子有效性证明”。

---

## 5. 护栏：数据、追踪、过拟合

### 5.1 Great Expectations

在数据入口做断言：

- 字段完整；
- 时间索引单调；
- 价格/成交量非负；
- 缺失率/异常值在阈值内。

防止 PIT 错误被 RD-Agent 循环放大。

### 5.2 MLflow

记录每轮：

- 因子表达式/代码版本；
- 数据快照；
- 参数/指标；
- 回放配置。

没有 MLflow，循环结果不可审计。

### 5.3 DSR/PBO

见第 4 节；位置固定在 Alphalens 之后、fincore 之前。

### 5.4 顺序

```
Great Expectations → RD-Agent → KunQuant/TA-Lib → Alphalens
→ DSR/PBO → fincore → WT-plus 回放
        ↑________________ MLflow ________________|
```

---

## 6. WT-plus 原生回放：最后一步

### 6.1 它做什么

- 用历史行情逐笔撮合；
- 计费用/滑点；
- 维护持仓/现金/权益；
- 输出委托/成交/拒单/账户。

### 6.2 它不做什么

- 不生成因子；
- 不证明因子有效；
- 不模拟柜台队列/网络延迟（这些是模型假设）。

### 6.3 记录字段

`status`、`completed_days`、`reason`、事件 ID、拒单原因、版本/交易日/成本场景。对账时按“版本+交易日+成本场景”绑定，不能只按日期合并。

---

## 7. 真实代码地图：本地 WT-plus 工具链怎么跑

这一节不是“概念图”，而是按本地仓库里的实际文件写的。路径：

`F:\XiangMu\Trader\wondertrader-master\.codex-work\factor-toolchain-20261030`

### 7.1 入口文件

| 文件 | 入口/函数 | 作用 |
|---|---|---|
| `factor_automation_worker.py` | 协调器 | 读资料、抽取/复核公式、提交研究、记录 SQLite |
| `factor_report_queue.py`（factor-chain/research-queue 版） | 队列 | 研究队列/报告状态 |
| `FactorToolchain.py` | `quality` / `BoundedSearch` / `standard_baselines` / `screen_panel` / `native_statistics` / `analyze_frozen` | 数据校验、Optuna、TA-Lib、Alphalens、DSR/PBO+fincore、MLflow |
| `factor_compute.py` | `compute(config, payload)` | 子进程调用 `kunquant_runner.py` |
| `kunquant_runner.py` | `run(payload)` | 隔离执行：`native_statistics`、`lifecycle`、`VerifiedEvaluator`/`analyze` |
| `rd_factor_agent.py` | `run(request)` | RD-Agent 假设生成 + AgentBridge/JSON schema + 每日调用预算 |
| `factor_native_audit.py` | `audit_version` | 读取原生回放收据、对齐 40 天、调用 DSR/PBO |
| `portfolio_worker.py` | `native_outcome` | 原生账户结果/费用/持仓 |

### 7.2 真实调用链

1. `factor_automation_worker.py` 从 `report.json` 抽公式，独立复核，`automation.submit(...)`。
2. `FactorToolchain.quality(frozen)` 用 Great Expectations 校验字段/时间/OHLC 包络。
3. `BoundedSearch` 用 Optuna TPE 搜窗口；预算上限 `min(6, budget)`，目标只用**验证集 oriented rank IC**。
4. `factor_compute.compute(config, payload)` 起子进程：
   ```python
   subprocess.run([python, '-I', str(runner)], input=json.dumps(payload),
                  capture_output=True, text=True, timeout=240,
                  env=dict(os.environ, OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1'))
   ```
5. `kunquant_runner.run` 里：`VerifiedEvaluator(frozen['rows'], frozen['spec'], frozen['data_snapshot'])` + `analyze(...)`；带 `toolchain` 时走 `FactorToolchain.analyze_frozen`。
6. `screen_panel` 用 Alphalens 的 `factor_information_coefficient` / `mean_return_by_quantile` / `quantile_turnover`；要求每个时间戳至少 5 个真实资产。
7. 原生回放：`factor_native_audit.audit_version` 读 `native-trials.json`，要求完整试验族、同一版本、同一数据快照、40 个共同交易日。
8. `native_statistics` 用 `holdout.deflated.deflate_trials` + `holdout.pbo.probability_of_backtest_overfitting`；门槛 DSR ≥ 0.95、PBO ≤ 0.10；通过后用 `fincore.metrics` 算收益/回撤/Sharpe。
9. `analyze_frozen` 用 MLflow（SQLite tracking）记录参数/指标。

### 7.3 四个容易被误解的点

1. **Alphalens 不是万能的**：单合约时序在 `screen_panel` 会返回 `not_applicable`（没有 5 个真实资产），不要硬造资产。
2. **DSR/PBO 不吃 IC**：`native_statistics` 要的是原生账户净收益矩阵，不是 IC 标签。
3. **原生回放要完整族**：`factor_native_audit` 要求所有尝试过的表达式/窗口，不是只交冠军；缺一个都不能“交集掉”。
4. **40 天是硬门槛**：少于 40 个共同完整交易日，直接 `waiting_aligned_native_days`。

### 7.4 读集成结果

本地 `integration.json` 里能看到真实状态：

- `engine`: KunQuant 0.1.12，185 bars，3 candidates，`max_absolute_error` 8.7e-19；
- `data_quality`: Great Expectations，4 checks；
- `baselines`: TA-Lib RSI/MACD/BBANDS 的 rank IC；
- `search`: Optuna TPE，6 trials，目标 validation oriented rank IC；
- `alphalens`: `not_applicable`（单合约）；
- `multiple_testing`: `waiting_native_trial_panel`；
- `mlflow_run_id`: 真实 run id。

**这张结果本身就是教材**：单合约因子在 Alphalens 上不适用、DSR/PBO 还在等原生面板——不是“跑完就算有效”。

---

## 8. 研究哲学：假设驱动 vs 数据驱动

RD-Agent 这类工具把“数据驱动发现”工业化：批量生成候选、批量回测、批量筛选。但它不替代经济逻辑，反而让多重检验更严重。正确分工是：

- **人**：提出假设、设定约束、判断经济机制、决定是否上线；
- **机器**：在框架内生成/检验/优化（RD-Agent + Optuna）；
- **统计护栏**：DSR/PBO 校正多轮搜索；
- **四道门**：逻辑、样本、交易、风险。

顶级 QR 的工作也不是“搜公式”，而是“假设→验证→解释失败→上线→监控”的循环；算力集群是把这个循环加速的基础设施，不是 Alpha 本身。

---

## 本章小结

### 一张表

| 层 | 工具 | 防什么 |
|---|---|---|
| 数据 | Great Expectations | 脏数据/PIT |
| 研究 | RD-Agent + Optuna | 搜索不足/参数 |
| 计算 | KunQuant + TA-Lib | 速度/基线 |
| 初筛 | Alphalens | IC/分组/换手 |
| 护栏 | DSR/PBO | 过拟合/多重检验 |
| 组合 | fincore | 绩效/归因 |
| 追踪 | MLflow | 不可复现 |
| 回放 | WT-plus | 执行/账户 |

### 八句话

1. WT-plus 是执行平台，不是因子研究器。
2. RD-Agent 扩大搜索，不保证 alpha。
3. Optuna 调参数，LLM 改结构。
4. KunQuant 快，但输入要 8 的倍数。
5. TA-Lib 是基线，不是答案。
6. Alphalens 是第一道筛，不是结论。
7. DSR/PBO 是过拟合护栏。
8. WT-plus 回放回答“账户会怎样”。

---

## 本章习题

**1.** WT-plus 的两个仓库分别是什么？

**2.** RD-Agent 的循环四步是什么？

**3.** Optuna 在循环里调什么？

**4.** KunQuant 对输入数量有什么要求？

**5.** Alphalens 主要看哪四类指标？

**6.** DSR/PBO 放在流水线的什么位置？

**7.** Great Expectations 和 MLflow 各防什么？

**8.** WT-plus 回放和普通回测的区别？

<details><summary>参考答案</summary>

**1.** WonderTrader-plus（C++ 核心）与 wtpy-plus（Python 封装）。

**2.** 假设→代码→回测→反馈。

**3.** 参数/超参（TPE）。

**4.** 输入数量必须是 8 的倍数。

**5.** IC/ICIR、分组收益、换手、暴露。

**6.** Alphalens 之后、fincore 之前。

**7.** GE 防脏数据/PIT；MLflow 防不可追溯/不可复现。

**8.** 回放用撮合/费用/持仓/账户语义逐笔执行；回测只是策略收益曲线。

</details>

---

## 参考资料

- WonderTrader-plus: <https://github.com/newbigdeng/WonderTrader-plus>
- wtpy-plus: <https://github.com/newbigdeng/wtpy-plus>
- RD-Agent: <https://github.com/microsoft/RD-Agent>
- KunQuant: <https://github.com/Menooker/KunQuant>
- TA-Lib: <https://github.com/TA-Lib/ta-lib-python>
- Alphalens-reloaded: <https://github.com/stefan-jansen/alphalens-reloaded>
- fincore: <https://github.com/cloudQuant/fincore>
- Optuna: <https://github.com/optuna/optuna>
- Great Expectations: <https://github.com/great-expectations/great_expectations>
- MLflow: <https://github.com/mlflow/mlflow>
- FactorZen: <https://github.com/rookiewu417/FactorZen>
